1. Imports + Load Dataset

In [63]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)

df = pd.read_csv("../data/WA_Fn-UseC_-Telco-Customer-Churn.csv")

print(df.head())
print("Shape:", df.shape)

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

2. Data Inspection + Cleaning

In [64]:
print("Columns:")
print(df.columns.tolist())

print("\nData Types:")
print(df.dtypes)

print("\nDuplicate Rows:", df.duplicated().sum())

print("\nMissing Values:")
print(df.isnull().sum())

print("\nStatistics:")
print(df.describe(include="all"))

print("\nChurn Distribution:")
print(df["Churn"].value_counts())

print("\nChurn Percentage:")
print(df["Churn"].value_counts(normalize=True) * 100)

# Convert TotalCharges to numeric
df["TotalCharges"] = pd.to_numeric(
    df["TotalCharges"],
    errors="coerce"
)

# Remove customer ID
df = df.drop("customerID", axis=1)

Columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Data Types:
customerID              str
gender                  str
SeniorCitizen         int64
Partner                 str
Dependents              str
tenure                int64
PhoneService            str
MultipleLines           str
InternetService         str
OnlineSecurity          str
OnlineBackup            str
DeviceProtection        str
TechSupport             str
StreamingTV             str
StreamingMovies         str
Contract                str
PaperlessBilling        str
PaymentMethod           str
MonthlyCharges      float64
TotalCharges            str
Churn                   str
dtype: object

Duplicate Rows: 0

Missing Values:
customerID      

3. X/y + Train/Validation/Test Split

In [65]:
X = df.drop("Churn", axis=1)

y = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

Training: (4930, 19)
Validation: (1056, 19)
Testing: (1057, 19)


4. Preprocessing

In [66]:
numerical_columns = [
    "SeniorCitizen",
    "tenure",
    "MonthlyCharges",
    "TotalCharges"
]

categorical_columns = [
    "gender",
    "Partner",
    "Dependents",
    "PhoneService",
    "MultipleLines",
    "InternetService",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Contract",
    "PaperlessBilling",
    "PaymentMethod"
]

numerical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("num", numerical_pipeline, numerical_columns),
    ("cat", categorical_pipeline, categorical_columns)
])

# Check preprocessing
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Validation shape:", X_val_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (4930, 45)
Validation shape: (1056, 45)
Testing shape: (1057, 45)


5. Logistic Regression

In [67]:
logistic_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

start = time.perf_counter()
logistic_pipeline.fit(X_train, y_train)
logistic_train_time = time.perf_counter() - start

start = time.perf_counter()
logistic_pred = logistic_pipeline.predict(X_val)
logistic_inference_time = time.perf_counter() - start

logistic_prob = logistic_pipeline.predict_proba(X_val)[:, 1]

print("Training time:", logistic_train_time)
print("Inference time:", logistic_inference_time)
print("Accuracy :", accuracy_score(y_val, logistic_pred))
print("Precision:", precision_score(y_val, logistic_pred))
print("Recall   :", recall_score(y_val, logistic_pred))
print("F1-score :", f1_score(y_val, logistic_pred))
print("ROC-AUC  :", roc_auc_score(y_val, logistic_prob))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, logistic_pred))

Training time: 0.13299643399659544
Inference time: 0.014522484998451546
Accuracy : 0.8058712121212122
Precision: 0.6459143968871596
Recall   : 0.5928571428571429
F1-score : 0.6182495344506518
ROC-AUC  : 0.8453124999999999

Confusion Matrix:
[[685  91]
 [114 166]]


6. KNN — K=3

In [68]:
knn3_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=3))
])

start = time.perf_counter()
knn3_pipeline.fit(X_train, y_train)
knn3_train_time = time.perf_counter() - start

start = time.perf_counter()
knn3_pred = knn3_pipeline.predict(X_val)
knn3_inference_time = time.perf_counter() - start

knn3_prob = knn3_pipeline.predict_proba(X_val)[:, 1]

print("Training time:", knn3_train_time)
print("Inference time:", knn3_inference_time)
print("Accuracy :", accuracy_score(y_val, knn3_pred))
print("Precision:", precision_score(y_val, knn3_pred))
print("Recall   :", recall_score(y_val, knn3_pred))
print("F1-score :", f1_score(y_val, knn3_pred))
print("ROC-AUC  :", roc_auc_score(y_val, knn3_prob))

Training time: 0.07429802200203994
Inference time: 0.12914425300550647
Accuracy : 0.7490530303030303
Precision: 0.5244299674267101
Recall   : 0.575
F1-score : 0.5485519591141397
ROC-AUC  : 0.7525358983799706


7. KNN — K=7

In [70]:
knn7_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", KNeighborsClassifier(n_neighbors=7))
])

start = time.perf_counter()
knn7_pipeline.fit(X_train, y_train)
knn7_train_time = time.perf_counter() - start

start = time.perf_counter()
knn7_pred = knn7_pipeline.predict(X_val)
knn7_inference_time = time.perf_counter() - start

knn7_prob = knn7_pipeline.predict_proba(X_val)[:, 1]

print("Training time:", knn7_train_time)
print("Inference time:", knn7_inference_time)
print("Accuracy :", accuracy_score(y_val, knn7_pred))
print("Precision:", precision_score(y_val, knn7_pred))
print("Recall   :", recall_score(y_val, knn7_pred))
print("F1-score :", f1_score(y_val, knn7_pred))
print("ROC-AUC  :", roc_auc_score(y_val, knn7_prob))

Training time: 0.08162030100356787
Inference time: 0.031042484006320592
Accuracy : 0.7727272727272727
Precision: 0.5684931506849316
Recall   : 0.5928571428571429
F1-score : 0.5804195804195804
ROC-AUC  : 0.8060866163475701


8. Decision Tree — Shallow

In [71]:
dt_shallow_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=3,
        random_state=42
    ))
])

start = time.perf_counter()
dt_shallow_pipeline.fit(X_train, y_train)
dt_shallow_train_time = time.perf_counter() - start

start = time.perf_counter()
dt_shallow_pred = dt_shallow_pipeline.predict(X_val)
dt_shallow_inference_time = time.perf_counter() - start

dt_shallow_prob = dt_shallow_pipeline.predict_proba(X_val)[:, 1]

dt_shallow_train_pred = dt_shallow_pipeline.predict(X_train)

print("Training time:", dt_shallow_train_time)
print("Inference time:", dt_shallow_inference_time)
print("Accuracy :", accuracy_score(y_val, dt_shallow_pred))
print("Precision:", precision_score(y_val, dt_shallow_pred))
print("Recall   :", recall_score(y_val, dt_shallow_pred))
print("F1-score :", f1_score(y_val, dt_shallow_pred))
print("ROC-AUC  :", roc_auc_score(y_val, dt_shallow_prob))

print("Training Accuracy:",
      accuracy_score(y_train, dt_shallow_train_pred))

print("Validation Accuracy:",
      accuracy_score(y_val, dt_shallow_pred))

Training time: 0.110249894001754
Inference time: 0.018175277997215744
Accuracy : 0.7935606060606061
Precision: 0.7012987012987013
Recall   : 0.38571428571428573
F1-score : 0.4976958525345622
ROC-AUC  : 0.8246203055964654
Training Accuracy: 0.7914807302231237
Validation Accuracy: 0.7935606060606061


9. Decision Tree — Deep

In [72]:
dt_deep_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", DecisionTreeClassifier(
        max_depth=15,
        random_state=42
    ))
])

start = time.perf_counter()
dt_deep_pipeline.fit(X_train, y_train)
dt_deep_train_time = time.perf_counter() - start

start = time.perf_counter()
dt_deep_pred = dt_deep_pipeline.predict(X_val)
dt_deep_inference_time = time.perf_counter() - start

dt_deep_prob = dt_deep_pipeline.predict_proba(X_val)[:, 1]

dt_deep_train_pred = dt_deep_pipeline.predict(X_train)

print("Training time:", dt_deep_train_time)
print("Inference time:", dt_deep_inference_time)
print("Accuracy :", accuracy_score(y_val, dt_deep_pred))
print("Precision:", precision_score(y_val, dt_deep_pred))
print("Recall   :", recall_score(y_val, dt_deep_pred))
print("F1-score :", f1_score(y_val, dt_deep_pred))
print("ROC-AUC  :", roc_auc_score(y_val, dt_deep_prob))

print("Training Accuracy:",
      accuracy_score(y_train, dt_deep_train_pred))

print("Validation Accuracy:",
      accuracy_score(y_val, dt_deep_pred))

Training time: 0.13430513099592645
Inference time: 0.01820380000572186
Accuracy : 0.7244318181818182
Precision: 0.48
Recall   : 0.4714285714285714
F1-score : 0.4756756756756757
ROC-AUC  : 0.6637564432989691
Training Accuracy: 0.9669371196754564
Validation Accuracy: 0.7244318181818182


10. Random Forest

In [73]:
rf_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=100,
        random_state=42
    ))
])

start = time.perf_counter()
rf_pipeline.fit(X_train, y_train)
rf_train_time = time.perf_counter() - start

start = time.perf_counter()
rf_pred = rf_pipeline.predict(X_val)
rf_inference_time = time.perf_counter() - start

rf_prob = rf_pipeline.predict_proba(X_val)[:, 1]

rf_train_pred = rf_pipeline.predict(X_train)

print("Training time:", rf_train_time)
print("Inference time:", rf_inference_time)
print("Accuracy :", accuracy_score(y_val, rf_pred))
print("Precision:", precision_score(y_val, rf_pred))
print("Recall   :", recall_score(y_val, rf_pred))
print("F1-score :", f1_score(y_val, rf_pred))
print("ROC-AUC  :", roc_auc_score(y_val, rf_prob))

print("Training Accuracy:",
      accuracy_score(y_train, rf_train_pred))

print("Validation Accuracy:",
      accuracy_score(y_val, rf_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_val, rf_pred))

Training time: 0.8461558730050456
Inference time: 0.05307975500181783
Accuracy : 0.7784090909090909
Precision: 0.5991379310344828
Recall   : 0.49642857142857144
F1-score : 0.54296875
ROC-AUC  : 0.8126610824742269
Training Accuracy: 0.9979716024340771
Validation Accuracy: 0.7784090909090909

Confusion Matrix:
[[683  93]
 [141 139]]


11. Gradient Boosting

In [74]:
gb_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", GradientBoostingClassifier(
        n_estimators=100,
        random_state=42
    ))
])

start = time.perf_counter()
gb_pipeline.fit(X_train, y_train)
gb_train_time = time.perf_counter() - start

start = time.perf_counter()
gb_pred = gb_pipeline.predict(X_val)
gb_inference_time = time.perf_counter() - start

gb_prob = gb_pipeline.predict_proba(X_val)[:, 1]

gb_train_pred = gb_pipeline.predict(X_train)

print("Training time:", gb_train_time)
print("Inference time:", gb_inference_time)
print("Accuracy :", accuracy_score(y_val, gb_pred))
print("Precision:", precision_score(y_val, gb_pred))
print("Recall   :", recall_score(y_val, gb_pred))
print("F1-score :", f1_score(y_val, gb_pred))
print("ROC-AUC  :", roc_auc_score(y_val, gb_prob))

print("Training Accuracy:",
      accuracy_score(y_train, gb_train_pred))

print("Validation Accuracy:",
      accuracy_score(y_val, gb_pred))

Training time: 1.2322483070020098
Inference time: 0.017217381995578762
Accuracy : 0.8011363636363636
Precision: 0.65625
Recall   : 0.525
F1-score : 0.5833333333333334
ROC-AUC  : 0.8478691089837996
Training Accuracy: 0.8318458417849899
Validation Accuracy: 0.8011363636363636
